# Notebook 2.5: 3D Vessel Segmentation

**Pipeline position**: After deconvolution (step 3), before EDF (step 4).  
**Input**: Deconvolved z-stacks from `data/processed/deconvolved/`  
**Output**: Vessel masks, skeletons, and morphometric features in `data/processed/vessel_3d/`

This notebook segments blood and lymphatic vessels from 3D deconvolved z-stacks using:
1. **Frangi vesselness filter** — multi-scale tubular structure enhancement
2. **Morphological cleanup** — threshold, small object removal, hole filling
3. **3D skeletonization** — Lee (1994) thinning to 1-voxel centerlines
4. **Graph-based morphometry** — per-segment length, diameter, tortuosity

The deconvolved z-stack is used because it retains full 3D volumetric information
with improved optical resolution, while EDF collapses the z-dimension.

**Vessel markers**: CD31 (blood endothelium), CD34, Lyve1 (lymphatic), SMActin, CollagenIV, Vimentin

## Section 1: Setup and Data Loading

In [ ]:
# Autoreload for development
%load_ext autoreload
%autoreload 2

import sys
import json
import numpy as np
from pathlib import Path

# Project setup
PROJECT_DIR = Path('.').resolve()
# Uncomment and modify if running from a different location:
# PROJECT_DIR = Path('/path/to/my_project')

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

from kintsugi.vessel3d import (
    VesselSpacing,
    VesselSegmentationResult,
    preprocess_volume,
    compute_vesselness_frangi,
    binarize_vessel_mask,
    skeletonize_vessels,
    prune_skeleton,
    analyze_vessel_graph,
    export_vessel_results,
    segment_vessels_3d,
)
from kintsugi.vessel3d_viz import (
    ortho_view,
    overlay_mask_on_raw,
    render_skeleton_mip,
    plot_vessel_features,
)

import logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

print(f"Project: {PROJECT_DIR}")

In [ ]:
# Load experiment metadata
experiment_path = PROJECT_DIR / 'meta' / 'experiment.json'
if experiment_path.exists():
    with open(experiment_path) as f:
        experiment_config = json.load(f)
    spacing = VesselSpacing.from_experiment(experiment_config)
    print(f"Loaded experiment.json")
else:
    experiment_config = {}
    spacing = VesselSpacing()  # Defaults: 377 nm XY, 1500 nm Z
    print("Using default spacing")

print(f"Spacing: XY={spacing.xy} um, Z={spacing.z} um, ratio={spacing.ratio:.2f}")

# Load channel names
try:
    from Kio import load_channel_names
    channels_per_cycle = experiment_config.get('channels_per_cycle', 4)
    channel_name_dict = load_channel_names(PROJECT_DIR / 'meta', channels_per_cycle=channels_per_cycle)
    print(f"Channel names loaded: {sum(len(v) for v in channel_name_dict.values())} total")
except Exception as e:
    channel_name_dict = {}
    print(f"Channel names not available: {e}")

In [ ]:
# === SELECT VESSEL MARKER AND CYCLE ===
# Modify these to match your panel:

VESSEL_CYCLE = 2         # Cycle containing the vessel marker
VESSEL_CHANNEL = 2       # Channel number (1-indexed)
VESSEL_MARKER = 'CD31'   # Marker name for file naming

# Processing device
DEVICE = 'auto'  # 'auto', 'gpu', or 'cpu'

print(f"Target: Cycle {VESSEL_CYCLE}, Channel {VESSEL_CHANNEL} ({VESSEL_MARKER})")

In [ ]:
# Load the deconvolved z-stack
decon_dir = PROJECT_DIR / 'data' / 'processed' / 'deconvolved' / f'cyc{VESSEL_CYCLE:02d}' / f'CH{VESSEL_CHANNEL}'

if not decon_dir.exists():
    raise FileNotFoundError(f"Deconvolution output not found: {decon_dir}")

# Load z-planes
from skimage.io import imread
from natsort import natsorted

tiff_files = natsorted(list(decon_dir.glob('*.tif')))
print(f"Found {len(tiff_files)} z-planes in {decon_dir}")

# Parallel loading for speed
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as executor:
    slices = list(executor.map(lambda f: imread(str(f)), tiff_files))

volume = np.stack(slices, axis=0)
del slices  # Free memory

print(f"Volume loaded: {volume.shape}, dtype={volume.dtype}")
print(f"Size: {volume.nbytes / 1e9:.2f} GB")
print(f"Intensity range: [{volume.min()}, {volume.max()}]")

In [ ]:
# Verify: inspect the loaded volume
fig = ortho_view(volume, title=f"{VESSEL_MARKER} - Raw Deconvolved")
fig.savefig(str(PROJECT_DIR / 'data' / 'processed' / 'vessel_3d' / f'{VESSEL_MARKER}_raw_ortho.png'),
            dpi=100, bbox_inches='tight')

## Section 2: Preprocessing — Anisotropy Correction and Denoising

The z-stack has ~4:1 anisotropy (Z voxels are ~4x larger than XY). Resampling
the z-axis to isotropic spacing is required for correct skeletonization and
radius estimation.

In [ ]:
# Preprocess: isotropic resampling + light denoising
preprocessed = preprocess_volume(
    volume,
    spacing=spacing,
    denoise_sigma=0.5,       # Light Gaussian, 0 to skip
    make_isotropic=True,     # Upsample z to match xy
    device=DEVICE,
)

print(f"Original shape: {volume.shape}")
print(f"Isotropic shape: {preprocessed.shape}")
print(f"Z resampled by {spacing.ratio:.2f}x")

In [ ]:
# Verify: inspect XZ cross-section before/after
import matplotlib.pyplot as plt

y_mid = volume.shape[1] // 2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(volume[:, y_mid, :], cmap='gray', aspect='auto')
axes[0].set_title(f'XZ Before (anisotropic, z={volume.shape[0]})')
axes[0].set_ylabel('Z')

y_mid_iso = preprocessed.shape[1] // 2
axes[1].imshow(preprocessed[:, y_mid_iso, :], cmap='gray', aspect='equal')
axes[1].set_title(f'XZ After (isotropic, z={preprocessed.shape[0]})')
axes[1].set_ylabel('Z')

fig.tight_layout()
plt.show()

## Section 3: Vessel Probability Map — Frangi Vesselness Filter

The Frangi filter enhances tubular structures by analyzing Hessian eigenvalues
at multiple Gaussian scales. High response = tube-like structure.

**Scales** should span the expected vessel radius range in isotropic voxels.
For typical fluorescence: sigmas=[1, 2, 4, 8] covers capillaries through
larger vessels.

In [ ]:
# Compute Frangi vesselness
vesselness = compute_vesselness_frangi(
    preprocessed,
    sigmas=[1.0, 2.0, 4.0, 8.0],  # Multi-scale vessel radii
    alpha=0.5,                      # Plate sensitivity
    beta=0.5,                       # Blob sensitivity
    gamma=None,                     # Auto (half max Hessian norm)
    spacing=spacing,
    device=DEVICE,
)

print(f"Vesselness range: [{vesselness.min():.4f}, {vesselness.max():.4f}]")
print(f"Non-zero fraction: {(vesselness > 0).sum() / vesselness.size:.4f}")

In [ ]:
# Verify: compare raw vs vesselness
fig = ortho_view(vesselness, title=f"{VESSEL_MARKER} - Frangi Vesselness", cmap='hot')
plt.show()

## Section 4: Binarization and Morphological Cleanup

Convert the vesselness map to a clean binary mask:
1. Otsu threshold (or manual)
2. Remove small objects (< min_size voxels)
3. Morphological closing (bridge small gaps)
4. Fill internal holes per z-plane

In [ ]:
# Interactive threshold exploration
from skimage.filters import threshold_otsu

nonzero = vesselness[vesselness > 0]
otsu_val = threshold_otsu(nonzero) if len(nonzero) > 0 else 0.1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of vesselness values
axes[0].hist(nonzero.ravel(), bins=200, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(otsu_val, color='red', linestyle='--', label=f'Otsu={otsu_val:.4f}')
axes[0].set_xlabel('Vesselness')
axes[0].set_ylabel('Count')
axes[0].set_title('Vesselness Histogram')
axes[0].legend()

# Preview binary mask at Otsu threshold
z_mid = vesselness.shape[0] // 2
axes[1].imshow(vesselness[z_mid] > otsu_val, cmap='gray')
axes[1].set_title(f'Binary preview (z={z_mid}, threshold={otsu_val:.4f})')

fig.tight_layout()
plt.show()

print(f"Otsu threshold: {otsu_val:.4f}")
print(f"Set THRESHOLD below to override, or None for Otsu auto-detection.")

In [ ]:
# Binarize and clean up
THRESHOLD = None  # Set to a float to override Otsu, e.g., 0.05
MIN_SIZE = 500    # Minimum vessel volume in voxels
CLOSING_RADIUS = 1  # Ball radius for morphological closing

mask = binarize_vessel_mask(
    vesselness,
    threshold=THRESHOLD,
    min_size=MIN_SIZE,
    closing_radius=CLOSING_RADIUS,
    fill_holes_2d=True,
)

print(f"Mask shape: {mask.shape}")
print(f"Vessel voxels: {mask.sum():,} ({100.0 * mask.sum() / mask.size:.2f}%)")

In [ ]:
# Verify: overlay mask on raw data
fig = overlay_mask_on_raw(preprocessed, mask, title=f"{VESSEL_MARKER} - Vessel Mask")
plt.show()

## Section 5: 3D Skeletonization

Reduce the binary mask to 1-voxel-wide centerlines using the Lee (1994)
thinning algorithm. Short terminal branches (spurs) are pruned.

**Memory note**: Skeletonization is CPU-only and operates on the full 3D array.
For the isotropic volume, this requires the array to fit in RAM.

In [ ]:
# Skeletonize
skeleton = skeletonize_vessels(mask)
print(f"Skeleton voxels: {skeleton.sum():,}")

# Prune short branches
PRUNE_MIN_LENGTH_UM = 5.0  # Minimum branch length in micrometers

skeleton_pruned = prune_skeleton(
    skeleton,
    min_branch_length_um=PRUNE_MIN_LENGTH_UM,
    spacing=spacing,
)
print(f"Pruned skeleton voxels: {skeleton_pruned.sum():,}")

In [ ]:
# Verify: skeleton MIP overlay
fig = render_skeleton_mip(
    skeleton_pruned,
    binary_mask=mask,
    title=f"{VESSEL_MARKER} - Skeleton (red) on Mask (gray)",
)
plt.show()

## Section 6: Graph Construction and Vessel Segment Analysis

Convert the skeleton to a graph (via skan) and extract per-segment features:
- **Length** (physical units, um)
- **Diameter** (from distance transform at skeleton voxels)
- **Tortuosity** (path length / Euclidean distance)
- **Branch type** (endpoint-endpoint, junction-endpoint, junction-junction)

In [ ]:
# Build graph and extract features
features = analyze_vessel_graph(
    skeleton_pruned,
    mask,
    spacing=spacing,
)

print(f"Total vessel segments: {len(features)}")
print(f"\nFeature summary:")
features[['branch_length_um', 'mean_diameter_um', 'tortuosity', 'branch_type']].describe()

In [ ]:
# Visualize morphometric features
fig = plot_vessel_features(
    features,
    title=f"{VESSEL_MARKER} - Vessel Morphometry",
)
plt.show()

## Section 7: Export Results

Save all outputs for downstream use:
1. `binary_mask_{marker}.tif` — 3D vessel mask (isotropic)
2. `binary_mask_{marker}_native.tif` — 3D mask at native resolution
3. `skeleton_{marker}.tif` — 3D skeleton
4. `vessel_features_{marker}.csv` — Per-segment morphometry
5. `vessel_graph_{marker}.graphml` — Graph for programmatic analysis
6. `vessel_2d_projection_{marker}.tif` — MIP for Notebook 4 spatial analysis

In [ ]:
# Create native-resolution mask
from scipy.ndimage import zoom as scipy_zoom
native_mask = scipy_zoom(mask.astype(np.float32), (1.0 / spacing.ratio, 1.0, 1.0), order=0) > 0.5

# Package results
result = VesselSegmentationResult(
    binary_mask=mask,
    binary_mask_native=native_mask,
    skeleton=skeleton_pruned,
    features=features,
    vesselness=vesselness,
    spacing=spacing,
    marker_name=VESSEL_MARKER,
)

# Export
output_dir = PROJECT_DIR / 'data' / 'processed' / 'vessel_3d'
outputs = export_vessel_results(
    result,
    output_dir=output_dir,
    save_vesselness=False,  # Set True to also save vesselness map (large)
)

print(f"\nAll outputs saved to: {output_dir}")
for name, path in outputs.items():
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    print(f"  {name}: {path.name} ({size_mb:.1f} MB)")

---

## Quick Pipeline (Alternative)

For a one-call pipeline that runs all steps automatically:

In [ ]:
# # Uncomment to run the full pipeline in one call:
# result = segment_vessels_3d(
#     volume,
#     spacing=spacing,
#     sigmas=[1.0, 2.0, 4.0, 8.0],
#     threshold=None,       # Otsu auto
#     min_size=500,
#     closing_radius=1,
#     denoise_sigma=0.5,
#     make_isotropic=True,
#     prune_min_length_um=5.0,
#     device='auto',
#     marker_name='CD31',
# )
# outputs = export_vessel_results(result, output_dir)